# Abstract-Free QA Datagen Correction

This notebook fills the missing `qa_no_abstract` category consumed by `pubmed_datagen_sft.ipynb`. It reuses validated clinical questions but never sends PubMed abstracts, titles, chunks, or prior answers to the generation model.

The output is educational model-training data, not clinical decision support.

## Execution Sequence

1. Configure paths and category targets.
2. Inspect the existing assembled and validated-QA inputs.
3. Review the derived output size and candidate distribution.
4. Generate question-only answers through OpenRouter.
5. Validate and export per-cancer-type ShareGPT JSONL files.
6. Run the existing SFT merge cell after review to incorporate this category.

## 1. Configure File Paths and Category Constants

The only generation input is `qa_validated/*.jsonl`. Source passages, titles, raw abstracts, and previous answers are intentionally excluded from the OpenRouter request.

In [1]:
import asyncio
import json
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

if Path("/workspace/training/pubmed").exists():
    PROJECT_ROOT = Path("/workspace/training/pubmed")
elif Path("/workspace/pubmed").exists():
    PROJECT_ROOT = Path("/workspace/pubmed")
else:
    PROJECT_ROOT = Path("/home/spark/projects/training/pubmed")

DATA_DIR = PROJECT_ROOT / "data"
TRAINING_DIR = DATA_DIR / "training-data"
OUTPUT_DIR = TRAINING_DIR / "pubmed_oncologist_v2"
ASSEMBLED_SFT_FILE = TRAINING_DIR / "pubmed_oncologist_v2.jsonl"
VALIDATED_QA_DIR = OUTPUT_DIR / "qa_validated"
NO_ABSTRACT_DIR = OUTPUT_DIR / "qa_no_abstract"
REJECTED_DIR = NO_ABSTRACT_DIR / "_rejected"
AUDIT_FILE = NO_ABSTRACT_DIR / "qa_no_abstract_audit.json"

QWEN36_SFT_NOTEBOOKS = [
    PROJECT_ROOT / "notebooks/loras/qwen36/pubmed_qwen36-27b-sft_NVFP4.ipynb",
    PROJECT_ROOT / "notebooks/loras/qwen36/pubmed_qwen36-27b-sft_bnb4bit.ipynb",
]

EXPECTED_CATEGORIES = {
    "qa": 0.35,
    "qa_no_abstract": 0.20,
    "treatment_reasoning": 0.12,
    "beyond_evidence": 0.05,
    "self_correction": 0.05,
    "continuation": 0.23,
}

API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
MODEL_NAME = "qwen/qwen3-235b-a22b-thinking-2507"
CONCURRENCY = 20
TEMPERATURE = 0.3
MAX_TOKENS = 3072
SEED = 3407

# None derives the full target from the existing assembled SFT QA count.
# Set a positive value for a bounded pilot run.
TARGET_COUNT = None

if not ASSEMBLED_SFT_FILE.exists():
    raise FileNotFoundError(f"Missing assembled SFT file: {ASSEMBLED_SFT_FILE}")
if not VALIDATED_QA_DIR.exists():
    raise FileNotFoundError(f"Missing validated QA directory: {VALIDATED_QA_DIR}")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is required to generate qa_no_abstract answers.")

NO_ABSTRACT_DIR.mkdir(parents=True, exist_ok=True)
REJECTED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Validated QA input: {VALIDATED_QA_DIR}")
print(f"Abstract-free output: {NO_ABSTRACT_DIR}")
print(f"Model: {MODEL_NAME}")

Project root: /workspace/training/pubmed
Validated QA input: /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2/qa_validated
Abstract-free output: /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2/qa_no_abstract
Model: qwen/qwen3-235b-a22b-thinking-2507


## 2. Load and Inspect Both Input Files

This section inspects the assembled SFT JSONL and the validated-QA source files. It reports schema and category counts without changing any data.

In [2]:
def read_jsonl(path: Path):
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if line:
                yield line_number, json.loads(line)

assembled_count = 0
assembled_categories = Counter()
assembled_keys = Counter()
assembled_sample = None
for _, row in read_jsonl(ASSEMBLED_SFT_FILE):
    assembled_count += 1
    assembled_categories[row.get("data_type", "qa")] += 1
    assembled_keys.update(row.keys())
    assembled_sample = assembled_sample or row

validated_files = sorted(VALIDATED_QA_DIR.glob("pubmed_*.jsonl"))
validated_counts = {}
validated_keys = Counter()
validated_sample = None
for file_path in validated_files:
    count = 0
    for _, row in read_jsonl(file_path):
        count += 1
        validated_keys.update(row.keys())
        validated_sample = validated_sample or row
    validated_counts[file_path.stem] = count

print(f"Assembled rows: {assembled_count:,}")
print(f"Assembled keys: {sorted(assembled_keys)}")
print(f"Assembled categories: {dict(assembled_categories)}")
print(f"Validated QA files: {len(validated_files)}")
print(f"Validated QA rows: {sum(validated_counts.values()):,}")
print(f"Validated QA keys: {sorted(validated_keys)}")
print(f"Validated QA by cancer type: {validated_counts}")
print(f"Assembled sample keys: {sorted(assembled_sample or {})}")
print(f"Validated-QA sample keys: {sorted(validated_sample or {})}")

Assembled rows: 32,243
Assembled keys: ['cancer_type', 'conversations', 'data_type']
Assembled categories: {'beyond_evidence': 14688, 'self_correction': 2941, 'qa': 14008, 'treatment_reasoning': 606}
Validated QA files: 10
Validated QA rows: 19,660
Validated QA keys: ['answer', 'cancer_type', 'chunk_key', 'conversations', 'grounding_verdict', 'question', 'round', 'source']
Validated QA by cancer type: {'pubmed_bone_cancer': 2076, 'pubmed_brain_tumour': 2262, 'pubmed_breast_cancer': 2072, 'pubmed_colon_cancer': 2071, 'pubmed_gastric_cancer': 1694, 'pubmed_kidney_cancer': 1721, 'pubmed_lung_cancer': 1928, 'pubmed_ovarian_cancer': 2177, 'pubmed_prostate_cancer': 1792, 'pubmed_skin_cancer': 1867}
Assembled sample keys: ['cancer_type', 'conversations', 'data_type']
Validated-QA sample keys: ['answer', 'cancer_type', 'chunk_key', 'conversations', 'grounding_verdict', 'question', 'round', 'source']


## 3. Compare File Structure, Schemas, and Record Content

The output category must be structurally compatible with the existing ShareGPT training file. This compares required conversation fields and reports fields that are intentionally not carried from validated QA into the OpenRouter request.

In [3]:
SHAREGPT_REQUIRED_KEYS = {"conversations", "data_type", "cancer_type"}
SHAREGPT_REQUIRED_ROLES = ["system", "human", "gpt"]

schema_report = {
    "assembled_required_keys_present": SHAREGPT_REQUIRED_KEYS.issubset(assembled_keys),
    "validated_keys": sorted(validated_keys),
    "output_required_keys": sorted(SHAREGPT_REQUIRED_KEYS),
    "source_fields_never_sent_to_model": [
        "abstract", "title", "passage", "chunk", "chunk_key", "answer", "conversations"
    ],
}

if assembled_sample:
    conversation = assembled_sample.get("conversations", [])
    schema_report["assembled_first_roles"] = [turn.get("from") for turn in conversation[:3]]
    schema_report["assembled_conversation_shape_valid"] = (
        [turn.get("from") for turn in conversation[:3]] == SHAREGPT_REQUIRED_ROLES
    )

print(json.dumps(schema_report, indent=2))

if not schema_report["assembled_required_keys_present"]:
    raise ValueError("The assembled SFT file is not ShareGPT-compatible.")

{
  "assembled_required_keys_present": true,
  "validated_keys": [
    "answer",
    "cancer_type",
    "chunk_key",
    "conversations",
    "grounding_verdict",
    "question",
    "round",
    "source"
  ],
  "output_required_keys": [
    "cancer_type",
    "conversations",
    "data_type"
  ],
  "source_fields_never_sent_to_model": [
    "abstract",
    "title",
    "passage",
    "chunk",
    "chunk_key",
    "answer",
    "conversations"
  ],
  "assembled_first_roles": [
    "system",
    "human",
    "gpt"
  ],
  "assembled_conversation_shape_valid": true
}


## 4. Identify the Missing Category and Affected Records

Validated QA rows provide a question and cancer type. They are candidates only; their prior answer and every source-context field are excluded from generation.

In [4]:
qa_count = assembled_categories.get("qa", 0)
if qa_count == 0:
    raise ValueError("Cannot derive a target without assembled qa rows.")

derived_total = round(qa_count / EXPECTED_CATEGORIES["qa"])
derived_target = round(derived_total * EXPECTED_CATEGORIES["qa_no_abstract"])
target_count = TARGET_COUNT if TARGET_COUNT is not None else derived_target

existing_output_questions = set()
existing_output_count = Counter()
for output_file in sorted(NO_ABSTRACT_DIR.glob("*.jsonl")):
    for _, row in read_jsonl(output_file):
        conversations = row.get("conversations", [])
        question = next((turn.get("value", "") for turn in conversations if turn.get("from") == "human"), "")
        if question:
            existing_output_questions.add(question.strip())
        existing_output_count[row.get("cancer_type", output_file.stem)] += 1

candidates_by_type = defaultdict(list)
invalid_candidates = []
for input_file in validated_files:
    default_cancer_type = input_file.stem
    for line_number, row in read_jsonl(input_file):
        question = row.get("question", "").strip()
        if not question and "conversations" in row:
            question = next((turn.get("value", "").strip() for turn in row["conversations"] if turn.get("from") == "human"), "")
        cancer_type = row.get("cancer_type", default_cancer_type)
        if len(question) < 15 or question in existing_output_questions:
            invalid_candidates.append({"file": str(input_file), "line": line_number, "reason": "missing_or_duplicate_question"})
            continue
        candidates_by_type[cancer_type].append(question)

available_candidates = sum(map(len, candidates_by_type.values()))
print(f"Existing qa_no_abstract rows: {sum(existing_output_count.values()):,}")
print(f"Derived full target: {derived_target:,}")
print(f"Requested target: {target_count:,}")
print(f"Available unique question candidates: {available_candidates:,}")
print(f"Skipped candidates: {len(invalid_candidates):,}")
print(f"Missing expected category in current assembled file: {'qa_no_abstract' not in assembled_categories}")

Existing qa_no_abstract rows: 0
Derived full target: 8,005
Requested target: 8,005
Available unique question candidates: 19,660
Skipped candidates: 0
Missing expected category in current assembled file: True


## 5. Create Corrected Data Generation Rules

Each model request contains only the oncology system instruction and one existing clinical question. The response must be self-contained and must not imply access to an abstract, study packet, tool, or retrieved source.

In [5]:
LEAKED_SOURCE_PHRASES = (
    "based on the abstract",
    "the provided abstract",
    "the abstract above",
    "the study above",
    "the research finding above",
    "the provided text",
    "the source text",
    "according to the abstract",
)

SYSTEM_PROMPT = """You are an oncology education assistant. Provide accurate, self-contained educational answers about cancer biology, diagnosis, and treatment.

Do not claim to have read a source, abstract, study, chart, guideline, tool result, or retrieved document. Do not invent study names, trial identifiers, statistics, cohort sizes, doses, biomarkers, or guideline recommendations. Distinguish established knowledge from uncertainty, and state when individualized clinical evaluation is needed. This response is educational and is not clinical decision support."""

USER_PROMPT = """Answer this clinical oncology question using general, well-established knowledge only. No source material is available to you.

Question: {question}

Write a direct, self-contained answer. Do not mention an abstract, a study above, a retrieved source, or a tool. Do not invent specifics when the question does not support them."""

def make_output_row(cancer_type: str, question: str, answer: str) -> dict:
    return {
        "conversations": [
            {"from": "system", "value": SYSTEM_PROMPT},
            {"from": "human", "value": question},
            {"from": "gpt", "value": answer},
        ],
        "data_type": "qa_no_abstract",
        "cancer_type": cancer_type,
        "provenance": {
            "generator": "pubmed_datagen_qa_no_abstract",
            "input": "validated_qa_question_only",
            "source_context_sent_to_model": False,
            "model": MODEL_NAME,
        },
    }

def validate_output_row(row: dict) -> list[str]:
    errors = []
    if not SHAREGPT_REQUIRED_KEYS.issubset(row):
        errors.append("missing_required_keys")
    turns = row.get("conversations", [])
    if [turn.get("from") for turn in turns] != SHAREGPT_REQUIRED_ROLES:
        errors.append("invalid_roles")
        return errors
    answer = turns[2].get("value", "").strip()
    if len(answer) < 100:
        errors.append("answer_too_short")
    lower_answer = answer.lower()
    if any(phrase in lower_answer for phrase in LEAKED_SOURCE_PHRASES):
        errors.append("leaked_source_reference")
    if any(phrase in lower_answer for phrase in ("as an ai", "as a language model")):
        errors.append("non_clinical_answer")
    if row.get("data_type") != "qa_no_abstract":
        errors.append("wrong_category")
    return errors

def allocate_proportional_quotas(total: int, pools: dict[str, list[str]]) -> dict[str, int]:
    available = sum(map(len, pools.values()))
    if total > available:
        raise ValueError(f"Requested {total:,} unique examples but only {available:,} candidates are available.")
    raw = {cancer_type: total * len(questions) / available for cancer_type, questions in pools.items()}
    quotas = {cancer_type: min(len(pools[cancer_type]), int(value)) for cancer_type, value in raw.items()}
    remaining = total - sum(quotas.values())
    for cancer_type in sorted(raw, key=lambda key: (raw[key] - int(raw[key]), len(pools[key])), reverse=True):
        if remaining == 0:
            break
        if quotas[cancer_type] < len(pools[cancer_type]):
            quotas[cancer_type] += 1
            remaining -= 1
    if remaining:
        raise ValueError("Could not allocate the requested number of unique candidates.")
    return quotas

async def generate_answer(client: AsyncOpenAI, semaphore: asyncio.Semaphore, question: str) -> str:
    async with semaphore:
        for attempt in range(3):
            try:
                response = await client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": USER_PROMPT.format(question=question)},
                    ],
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                )
                return (response.choices[0].message.content or "").strip()
            except Exception:
                if attempt == 2:
                    return ""
                await asyncio.sleep(2 ** attempt)
    return ""

## 6. Generate and Validate Missing-Category Examples

This cell selects unique validated questions proportionally by cancer type, generates answers without retrieval context, and writes one resumable output JSONL per cancer type. It is intentionally not executed by this agent.

In [6]:
random.seed(SEED)
quotas = allocate_proportional_quotas(target_count, candidates_by_type)
selected = []
for cancer_type, quota in quotas.items():
    selected.extend((cancer_type, question) for question in random.sample(candidates_by_type[cancer_type], quota))
random.shuffle(selected)

print(f"Selected {len(selected):,} unique questions across {len(quotas)} cancer types")
print(f"Quotas: {quotas}")

async def generate_one(client: AsyncOpenAI, semaphore: asyncio.Semaphore, cancer_type: str, question: str):
    answer = await generate_answer(client, semaphore, question)
    return cancer_type, question, answer

async def generate_dataset(selected_questions: list[tuple[str, str]]):
    client = AsyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY, max_retries=2, timeout=180.0)
    semaphore = asyncio.Semaphore(CONCURRENCY)
    accepted_by_type = defaultdict(list)
    rejected_by_type = defaultdict(list)
    completed = 0
    accepted = 0
    rejected = 0

    for batch_start in range(0, len(selected_questions), CONCURRENCY):
        batch = selected_questions[batch_start:batch_start + CONCURRENCY]
        tasks = [
            asyncio.create_task(generate_one(client, semaphore, cancer_type, question))
            for cancer_type, question in batch
        ]

        for task in asyncio.as_completed(tasks):
            cancer_type, question, answer = await task
            row = make_output_row(cancer_type, question, answer)
            errors = validate_output_row(row)
            completed += 1

            if errors:
                rejected_by_type[cancer_type].append({"question": question, "reason": errors})
                rejected += 1
                output_path = REJECTED_DIR / f"{cancer_type}_qa_no_abstract_rejected.jsonl"
                output_row = {"question": question, "reason": errors}
            else:
                accepted_by_type[cancer_type].append(row)
                accepted += 1
                output_path = NO_ABSTRACT_DIR / f"{cancer_type}_qa_no_abstract.jsonl"
                output_row = row

            with output_path.open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(output_row, ensure_ascii=False) + "\n")
                handle.flush()

            print(
                f"Progress: {completed:,}/{len(selected_questions):,} "
                f"({completed / len(selected_questions):.1%}) | "
                f"accepted: {accepted:,} | rejected: {rejected:,}",
                flush=True,
            )

    return accepted_by_type, rejected_by_type

accepted_by_type, rejected_by_type = await generate_dataset(selected)
print(f"Accepted: {sum(map(len, accepted_by_type.values())):,}")
print(f"Rejected: {sum(map(len, rejected_by_type.values())):,}")

Selected 8,005 unique questions across 10 cancer types
Quotas: {'pubmed_bone_cancer': 845, 'pubmed_brain_tumour': 921, 'pubmed_breast_cancer': 844, 'pubmed_colon_cancer': 843, 'pubmed_gastric_cancer': 690, 'pubmed_kidney_cancer': 701, 'pubmed_lung_cancer': 785, 'pubmed_ovarian_cancer': 886, 'pubmed_prostate_cancer': 730, 'pubmed_skin_cancer': 760}
Progress: 1/8,005 (0.0%) | accepted: 1 | rejected: 0
Progress: 2/8,005 (0.0%) | accepted: 2 | rejected: 0
Progress: 3/8,005 (0.0%) | accepted: 3 | rejected: 0
Progress: 4/8,005 (0.0%) | accepted: 4 | rejected: 0
Progress: 5/8,005 (0.1%) | accepted: 5 | rejected: 0
Progress: 6/8,005 (0.1%) | accepted: 6 | rejected: 0
Progress: 7/8,005 (0.1%) | accepted: 7 | rejected: 0
Progress: 8/8,005 (0.1%) | accepted: 8 | rejected: 0
Progress: 9/8,005 (0.1%) | accepted: 9 | rejected: 0
Progress: 10/8,005 (0.1%) | accepted: 10 | rejected: 0
Progress: 11/8,005 (0.1%) | accepted: 11 | rejected: 0
Progress: 12/8,005 (0.1%) | accepted: 12 | rejected: 0
Progress

## 7. Export the Corrected Dataset and Audit Report

After generation, validate all category rows, report duplicates and cancer distribution, and write a reproducible audit report. The existing SFT merge notebook remains responsible for creating the final combined training file.

In [7]:
output_rows = []
validation_errors = Counter()
question_counts = Counter()
category_counts = Counter()
cancer_counts = Counter()

for output_file in sorted(NO_ABSTRACT_DIR.glob("*.jsonl")):
    for line_number, row in read_jsonl(output_file):
        output_rows.append(row)
        category_counts[row.get("data_type", "missing")] += 1
        cancer_counts[row.get("cancer_type", "missing")] += 1
        question = next((turn.get("value", "").strip() for turn in row.get("conversations", []) if turn.get("from") == "human"), "")
        question_counts[question] += 1
        validation_errors.update(validate_output_row(row))

duplicate_questions = sorted(question for question, count in question_counts.items() if question and count > 1)
audit_report = {
    "category": "qa_no_abstract",
    "row_count": len(output_rows),
    "category_counts": dict(category_counts),
    "cancer_type_counts": dict(sorted(cancer_counts.items())),
    "duplicate_question_count": len(duplicate_questions),
    "validation_errors": dict(validation_errors),
    "generation_parameters": {
        "model": MODEL_NAME,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "seed": SEED,
        "target_count": target_count,
        "source_context_sent_to_model": False,
    },
}

with AUDIT_FILE.open("w", encoding="utf-8") as handle:
    json.dump(audit_report, handle, indent=2, ensure_ascii=False)

print(json.dumps(audit_report, indent=2))
if validation_errors or duplicate_questions:
    raise ValueError("qa_no_abstract validation failed; review the audit report and rejected rows.")

{
  "category": "qa_no_abstract",
  "row_count": 8000,
  "category_counts": {
    "qa_no_abstract": 8000
  },
  "cancer_type_counts": {
    "pubmed_bone_cancer": 845,
    "pubmed_brain_tumour": 920,
    "pubmed_breast_cancer": 841,
    "pubmed_colon_cancer": 843,
    "pubmed_gastric_cancer": 690,
    "pubmed_kidney_cancer": 701,
    "pubmed_lung_cancer": 785,
    "pubmed_ovarian_cancer": 886,
    "pubmed_prostate_cancer": 730,
    "pubmed_skin_cancer": 759
  },
  "duplicate_question_count": 0,
  "validation_errors": {},
  "generation_parameters": {
    "model": "qwen/qwen3-235b-a22b-thinking-2507",
    "temperature": 0.3,
    "max_tokens": 3072,
    "seed": 3407,
    "target_count": 8005,
    "source_context_sent_to_model": false
  }
}


## 8. Locate Qwen 3.6 SFT Notebook Category Dependencies

The two Qwen 3.6 notebooks are inspected as JSON. This checks whether their source filters, maps, samples, or reports depend on a fixed data-type taxonomy.

In [8]:
CATEGORY_TOKENS = ("data_type", "qa_no_abstract", "qa", "continuation", "treatment_reasoning")
trainer_dependency_report = {}

for notebook_path in QWEN36_SFT_NOTEBOOKS:
    if not notebook_path.exists():
        trainer_dependency_report[str(notebook_path)] = {"exists": False}
        continue
    with notebook_path.open(encoding="utf-8") as handle:
        notebook = json.load(handle)
    code = "\n".join(
        "".join(cell.get("source", []))
        for cell in notebook.get("cells", [])
        if cell.get("cell_type") == "code"
    )
    trainer_dependency_report[str(notebook_path)] = {
        "exists": True,
        "uses_combined_sharegpt_path": "pubmed_oncologist_combined_sharegpt.jsonl" in code,
        "mentions_data_type": "data_type" in code,
        "mentions_qa_no_abstract": "qa_no_abstract" in code,
        "maps_sharegpt_conversations": '"conversations"' in code and '"human": "user"' in code,
        "category_tokens_found": [token for token in CATEGORY_TOKENS if token in code],
    }

print(json.dumps(trainer_dependency_report, indent=2))

{
  "/workspace/training/pubmed/notebooks/loras/qwen36/pubmed_qwen36-27b-sft_NVFP4.ipynb": {
    "exists": true,
    "uses_combined_sharegpt_path": true,
    "mentions_data_type": false,
    "mentions_qa_no_abstract": false,
    "maps_sharegpt_conversations": true,
    "category_tokens_found": []
  },
  "/workspace/training/pubmed/notebooks/loras/qwen36/pubmed_qwen36-27b-sft_bnb4bit.ipynb": {
    "exists": true,
    "uses_combined_sharegpt_path": true,
    "mentions_data_type": false,
    "mentions_qa_no_abstract": false,
    "maps_sharegpt_conversations": true,
    "category_tokens_found": []
  }
}


## 9. Assess Required Changes for Both SFT Notebooks

No trainer change is required for `qa_no_abstract` when it is written as a normal ShareGPT conversation: both trainers already load `conversations` and map `system`/`human`/`gpt` to their chat roles. The only useful optional change is reporting the input `data_type` distribution so a restart can confirm the intended category was included.

In [9]:
sft_update_plan = {
    "required_changes": [],
    "not_required": [
        "data loading: qa_no_abstract uses the existing ShareGPT conversations schema",
        "role mapping: uses existing system/human/gpt mapping",
        "prompt formatting: no category-specific chat template is needed",
        "training configuration: SFTTrainer trains conversations regardless of data_type",
    ],
    "recommended_optional_change": [
        "Count and print data_type values while loading the combined JSONL.",
        "Fail fast before training if qa_no_abstract is expected but absent from the merged file.",
    ],
    "pre_restart_check": {
        "combined_file": str(OUTPUT_DIR / "pubmed_oncologist_combined_sharegpt.jsonl"),
        "expected_category": "qa_no_abstract",
        "expected_target_fraction": EXPECTED_CATEGORIES["qa_no_abstract"],
    },
}
print(json.dumps(sft_update_plan, indent=2))

{
  "required_changes": [],
  "not_required": [
    "data loading: qa_no_abstract uses the existing ShareGPT conversations schema",
    "role mapping: uses existing system/human/gpt mapping",
    "prompt formatting: no category-specific chat template is needed",
    "training configuration: SFTTrainer trains conversations regardless of data_type"
  ],
  "recommended_optional_change": [
    "Count and print data_type values while loading the combined JSONL.",
    "Fail fast before training if qa_no_abstract is expected but absent from the merged file."
  ],
  "pre_restart_check": {
    "combined_file": "/workspace/training/pubmed/data/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl",
    "expected_category": "qa_no_abstract",
    "expected_target_fraction": 0.2
  }
}


## 10. Apply and Validate SFT Notebook Updates

No Qwen 3.6 SFT notebook edit is applied by this correction notebook. Before restarting either trainer, run the following check against the rebuilt combined dataset and confirm that `qa_no_abstract` is present at the intended proportion.

In [10]:
COMBINED_SFT_FILE = OUTPUT_DIR / "pubmed_oncologist_combined_sharegpt.jsonl"
if not COMBINED_SFT_FILE.exists():
    print(f"Merge has not run yet: {COMBINED_SFT_FILE}")
else:
    combined_distribution = Counter()
    combined_rows = 0
    for _, row in read_jsonl(COMBINED_SFT_FILE):
        combined_rows += 1
        combined_distribution[row.get("data_type", "qa")] += 1

    actual_fraction = combined_distribution["qa_no_abstract"] / max(combined_rows, 1)
    print(f"Combined rows: {combined_rows:,}")
    print(f"Data-type distribution: {dict(combined_distribution)}")
    print(f"qa_no_abstract fraction: {actual_fraction:.2%}")
    print(f"Target fraction: {EXPECTED_CATEGORIES['qa_no_abstract']:.2%}")
    if combined_distribution["qa_no_abstract"] == 0:
        raise ValueError(
            "The combined dataset is stale or was merged before qa_no_abstract was generated. "
            "Run Section 9, 'Merge & Assemble Combined Training Data', in "
            "pubmed_datagen_sft.ipynb, then rerun this check. Do not restart SFT yet."
        )

Combined rows: 33,349
Data-type distribution: {'continuation': 11006, 'qa': 14007, 'treatment_reasoning': 5002, 'beyond_evidence': 1667, 'self_correction': 1667}
qa_no_abstract fraction: 0.00%
Target fraction: 20.00%


ValueError: Combined dataset is missing qa_no_abstract; do not restart SFT yet.